In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

In [0]:
cus_details = spark.table("workspace.bronze.erp_customer_details")

In [0]:
name_map = {
    "CID": "customer_number",
    "BDATE": "birth_date",
    "GEN": "gender"
}

for old_name, new_name in name_map.items():
    cus_details = cus_details.withColumnRenamed(old_name, new_name)

In [0]:
cus_details = cus_details.withColumn("customer_number", 
    F.when(col("customer_number").startswith("NAS"), 
           F.substring(col("customer_number"),4, F.length(col("customer_number"))))
        .otherwise(col("customer_number"))
)

In [0]:
cus_details = cus_details.withColumn("gender",
                   F.when(F.upper(col("gender")).isin("F", "FEMALE"), "Female")
                   .when(F.upper(col("gender")).isin("M", "MALE"), "Male")
                   .otherwise("n/a")
)

In [0]:

cus_details.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customer_details")